In [ ]:
import re 
import pandas as pd
from collections import Counter


In [ ]:
df = pd.read_csv("/Users/kaitaoyang/Downloads/datasets_conversations/DailyDialog/train.csv")
print(df.shape)
df.head(2)

In [ ]:
text_raw = " ".join(df.dialog.to_list()).lower()
text_clean_list = re.findall(r"[a-zA-Z\-]+", text_raw)
text_clean_list[:100]

In [ ]:
len(set(text_clean_list))

In [ ]:
word_counter = Counter(text_clean_list)
word_counter.most_common(1000)


In [ ]:
# pip install google-cloud-translate
from google.oauth2 import service_account
from google.cloud import translate_v2 as translate

# Load credentials from the JSON file
credentials = service_account.Credentials.from_service_account_file("../kt-languages-148ff3e33c54.json")

# Initialize client
translate_client = translate.Client(credentials=credentials)

# Example usage
result = translate_client.translate("Hello world", target_language="ru")
print(result["translatedText"])




In [ ]:
import argostranslate.package
import argostranslate.translate

# Update model index (list of all available models)
argostranslate.package.update_package_index()
available_packages = argostranslate.package.get_available_packages()

# Find the English → Russian package
package_to_install = next(
    pkg for pkg in available_packages
    if pkg.from_code == "en" and pkg.to_code == "ru"
)

# Install the package
argostranslate.package.install_from_path(package_to_install.download())

# Now load installed languages
installed_languages = argostranslate.translate.get_installed_languages()
from_lang = next(lang for lang in installed_languages if lang.code == "en")
to_lang = next(lang for lang in installed_languages if lang.code == "ru")

translation = from_lang.get_translation(to_lang)

# Test translation
print(translation.translate("Hello world, how are you?"))


In [ ]:
# Apply row by row, keeping partial results
df["dialog_ru"] = df["dialog"].apply(translation.translate)

# Save partial progress, so you don’t lose already translated results
df.to_csv("translated_partial.csv", index=False, encoding="utf-8-sig")

In [ ]:
df.to_csv("translated_partial_ru.csv", index=False, encoding="utf-8-sig")

df["dialog_ru"].iloc[0]

In [ ]:
df = pd.read_csv("translated_partial_ru.csv")
text_raw_ru = " ".join(df["dialog_ru"].to_list()).lower()
len(text_raw_ru)

In [ ]:
text_clean_ru_list = re.findall(r"[а-яё\-]+", text_raw_ru)
len(set(text_clean_ru_list))

In [ ]:
word_counter_ru = Counter(text_clean_ru_list)
word_counter_ru.most_common(1000)

In [ ]:
df_words_ru = pd.DataFrame(word_counter_ru.most_common(), columns=["word", "count"])
df_words_ru.sort_values(by="count", ascending=False, inplace=True)
df_words_ru

In [ ]:
def substring(s, n):
    if len(s) < n:
        return [s]
    return [s[i:i+n] for i in range(len(s)-n+1)]

print(substring("abcdefg", 3))




In [ ]:
for n in [3, 4, 5]:
    df_words_ru[f"substr{n}"] = df_words_ru["word"].apply(substring, n=n)
df_words_ru.tail(5)


In [ ]:
substr3_ru_counts = Counter(df_words_ru.substr3.sum())
print(len(substr3_ru_counts))
substr3_ru_counts.most_common()

In [ ]:
substr4_ru_counts = Counter(df_words_ru.substr4.sum())
print(len(substr4_ru_counts))
substr4_ru_counts.most_common()

In [ ]:
substr5_ru_counts = Counter(df_words_ru.substr5.sum())
print(len(substr5_ru_counts))
substr5_ru_counts.most_common()

In [ ]:
df_words_ru.query("count>=20").shape
df_words_ru.query("count>=20").word.tolist()

In [ ]:
# def get_counts(str_list, counter):
#     return [(s, counter.get(s)) for s in str_list if counter.get(s) > 1]

# df_words_ru["substr3_count"] = df_words_ru["substr3"].apply(get_counts, counter=substr3_ru_counts)
# df_words_ru["substr4_count"] = df_words_ru["substr4"].apply(get_counts, counter=substr4_ru_counts)
# df_words_ru["substr5_count"] = df_words_ru["substr5"].apply(get_counts, counter=substr5_ru_counts)
df_words_ru.tail(30)

In [ ]:
df_words_ru[df_words_ru.word.str.contains("отсут")].word.to_list()

In [ ]:
from collections import defaultdict

class TrieNode:
    def __init__(self):
        self.children = {}
        self.words = []  # words that pass through this node

class Trie:
    def __init__(self):
        self.root = TrieNode()
    
    def insert(self, word):
        node = self.root
        for char in word:
            if char not in node.children:
                node.children[char] = TrieNode()
            node = node.children[char]
            node.words.append(word)
    
    def get_prefix_groups(self, min_group_size=2):
        """Return all prefix groups with at least min_group_size words"""
        result = []
        
        def dfs(node, prefix):
            # Only consider nodes with enough words
            if len(node.words) >= min_group_size:
                # If all children have fewer words, this is the longest prefix
                if all(len(child.words) < min_group_size for child in node.children.values()):
                    result.append((prefix, node.words))
            for char, child in node.children.items():
                dfs(child, prefix + char)
        
        dfs(self.root, "")
        return result

# Example usage
words =  df_words_ru.word.to_list()

trie = Trie()
for word in words:
    trie.insert(word)

groups = trie.get_prefix_groups()
for prefix, group_words in groups:
    print(f"'{prefix}' -> {group_words}")
